In [1]:
import os
import pickle
import torch
from torch.utils.data import Dataset
import random

AA_TO_INDEX = {
    'A': 0, 'C': 1, 'D': 2, 'E': 3, 'F': 4,
    'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9,
    'M': 10, 'N': 11, 'P': 12, 'Q': 13, 'R': 14,
    'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19,
    '-': 20, 'X': 20  # padding 또는 unknown
}

def random_voxel_rotate(voxel):
    # voxel: Tensor [C, D, H, W]
    if random.random() < 0.5:  # 50% 확률로 회전 적용
        axes = [(2, 3), (1, 3), (1, 2)]  # (H, W), (D, W), (D, H)
        k = random.choice([1, 2, 3])  # 실제 회전만 (0 제외)
        axis = random.choice(axes)
        voxel = torch.rot90(voxel, k=k, dims=axis)
    return voxel

def random_voxel_flip(voxel):
    if random.random() < 0.5:
        voxel = torch.flip(voxel, dims=[1])  # D-axis flip
    if random.random() < 0.5:
        voxel = torch.flip(voxel, dims=[2])  # H-axis flip
    if random.random() < 0.5:
        voxel = torch.flip(voxel, dims=[3])  # W-axis flip
    return voxel

class VoxelDataset(Dataset):
    def __init__(self, df, voxel_cache_dir, aug=False):
        self.df = df.reset_index(drop=True)
        self.voxel_cache_dir = voxel_cache_dir
        self.aug = aug

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        uid = row["UniProtID"]
        mut_pos = row["MutPos(pdb)"]
        wt = row["WT"]
        mut = row["Mut"]
        label = row["Label"]

        key = f"{uid}_{mut_pos}"
        voxel_path = os.path.join(self.voxel_cache_dir, f"{key}.pkl")

        # Load voxel
        with open(voxel_path, "rb") as f:
            data = pickle.load(f)
            feature = data["feature"]  # shape: (1, 7, 7, 7, 63)

        # Preprocess
        feature_tensor = torch.from_numpy(feature).permute(0, 4, 1, 2, 3).float().squeeze(0)  # (63, 7, 7, 7)

        if self.aug:
            feature_tensor = random_voxel_rotate(feature_tensor)
            feature_tensor = random_voxel_flip(feature_tensor)

        # Convert WT/Mut AA to index
        ref_idx = torch.tensor(AA_TO_INDEX.get(str(wt), 20), dtype=torch.long)
        mut_idx = torch.tensor(AA_TO_INDEX.get(str(mut), 20), dtype=torch.long)

        return feature_tensor, ref_idx, mut_idx, torch.tensor(label).long()


In [2]:
from torch.utils.data import DataLoader
import pandas as pd
from sklearn.model_selection import KFold

df = pd.read_csv(r"/mnt/c/Users/Kunny/Research/Project/BiConVarNet/filtered_variants_cleaned_final.tsv", sep="\t", )

# 10-fold 
kf = KFold(n_splits=10, shuffle=True, random_state=42)
splits = list(kf.split(df))
train_idx, val_idx = splits[0]

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()

# oversampling: label == 1
pos_df = train_df[train_df["Label"] == 1]
neg_df = train_df[train_df["Label"] == 0]

repeat_factor = max(1, len(neg_df) // max(len(pos_df), 1))
oversampled_train_df = pd.concat([neg_df, pd.concat([pos_df] * repeat_factor)], ignore_index=True)
oversampled_train_df = oversampled_train_df.sample(frac=1, random_state=42).reset_index(drop=True)  # 셔플

In [3]:
from torch.utils.data import DataLoader

voxel_cache_dir = "/mnt/c/Users/Kunny/Research/Dataset/Missense_Variant_dataset/voxel_cache"

train_dataset = VoxelDataset(oversampled_train_df, voxel_cache_dir, aug=True)
val_dataset   = VoxelDataset(val_df, voxel_cache_dir)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4,
                          persistent_workers=True, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4,
                          persistent_workers=True, pin_memory=True)

In [4]:
import torch
import torch.nn as nn

class VoxelMBConvClassifier(nn.Module):
    def __init__(self, in_ch=63, emb_dim=128, dropout_p=0.3):
        super().__init__()
        self.backbone = nn.Sequential(
            MBConv3D(in_ch, 64, expand_ratio=6),
            MBConv3D(64, 64, expand_ratio=6),
            MBConv3D(64, emb_dim, expand_ratio=6)
        )
        self.pool = nn.AdaptiveAvgPool3d(1)  # → [B, 128, 1, 1, 1]
        self.classifier = nn.Sequential(
            nn.Flatten(),                             # → [B, 128]
            nn.Linear(emb_dim, emb_dim),
            nn.BatchNorm1d(emb_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_p),
            nn.Linear(emb_dim, 1),
            nn.Sigmoid()  # Binary classification
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x.squeeze(-1)


class SqueezeExcitation3D(nn.Module):
    def __init__(self, in_channels, reduction=24):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.se = nn.Sequential(
            nn.Conv3d(in_channels, in_channels // reduction, kernel_size=1),
            nn.SiLU(),
            nn.Conv3d(in_channels // reduction, in_channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        scale = self.se(self.pool(x))
        return x * scale

class MBConv3D(nn.Module):
    def __init__(self, in_ch, out_ch, expand_ratio=6, kernel_size=3, stride=1, se_reduction=24):
        super().__init__()
        mid_ch = in_ch * expand_ratio

        self.use_res_connect = (stride == 1 and in_ch == out_ch)

        self.expand = nn.Sequential(
            nn.Conv3d(in_ch, mid_ch, kernel_size=1, bias=False),
            nn.BatchNorm3d(mid_ch),
            nn.SiLU()
        ) if expand_ratio != 1 else nn.Identity()

        self.depthwise = nn.Sequential(
            nn.Conv3d(mid_ch, mid_ch, kernel_size=kernel_size, stride=stride,
                      padding=kernel_size//2, groups=mid_ch, bias=False),
            nn.BatchNorm3d(mid_ch),
            nn.SiLU()
        )

        self.se = SqueezeExcitation3D(mid_ch, reduction=se_reduction)

        self.project = nn.Sequential(
            nn.Conv3d(mid_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm3d(out_ch)
        )

    def forward(self, x):
        identity = x
        out = self.expand(x)
        out = self.depthwise(out)
        out = self.se(out)
        out = self.project(out)

        if self.use_res_connect:
            return out + identity
        else:
            return out

class VoxelMBConvClassifier(nn.Module):
    def __init__(self, in_ch=63, emb_dim=128, dropout_p=0.3):
        super().__init__()
        
        # 3D 구조 백본
        self.backbone = nn.Sequential(
            MBConv3D(in_ch, 32, expand_ratio=6),     # [7×7×7]
            MBConv3D(32, 32, expand_ratio=6),
            MBConv3D(32, 48, expand_ratio=6),
            MBConv3D(48, 48, expand_ratio=6),
            MBConv3D(48, 64, expand_ratio=6),
            MBConv3D(64, 64, expand_ratio=6, stride=2),  # 다운샘플링: → [4×4×4]
            MBConv3D(64, 64, expand_ratio=6),
            MBConv3D(64, 96, expand_ratio=6),
            MBConv3D(96, 96, expand_ratio=6),
            MBConv3D(96, 96, expand_ratio=6),
            MBConv3D(96, 96, expand_ratio=6),
            MBConv3D(96, 96, expand_ratio=6),
            MBConv3D(96, emb_dim, expand_ratio=6),
            MBConv3D(emb_dim, emb_dim, expand_ratio=6),
            MBConv3D(emb_dim, emb_dim, expand_ratio=6)
        )
        self.pool = nn.AdaptiveAvgPool3d(1)  # → [B, 128, 1, 1, 1]

        # Mutation Embedding (64 + 64 → 128)
        self.ref_emb = nn.Embedding(21, emb_dim // 2)  # 64
        self.mut_emb = nn.Embedding(21, emb_dim // 2)  # 64
        self.mut_fusion = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.BatchNorm1d(emb_dim),
            nn.SiLU(),
            nn.Linear(emb_dim, emb_dim),
            nn.BatchNorm1d(emb_dim),
            nn.SiLU()
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.BatchNorm1d(emb_dim),
            nn.SiLU(),
            nn.Dropout(dropout_p),
            nn.Linear(emb_dim, 2)  # Binary classification (logit)
        )

    def forward(self, x, ref_idx, mut_idx):
        x = self.backbone(x)               # [B, 128, 7, 7, 7]
        x = self.pool(x).squeeze(-1).squeeze(-1).squeeze(-1)  # → [B, 128]

        # Mutation embedding
        ref_vec = self.ref_emb(ref_idx)    # [B, 64]
        mut_vec = self.mut_emb(mut_idx)    # [B, 64]
        mut_feat = self.mut_fusion(torch.cat([ref_vec, mut_vec], dim=1))  # [B, 128]

        # Combine structure & mutation features
        x = x + mut_feat                   # [B, 128]

        return self.classifier(x)          # [B, 2]

In [5]:
from torchinfo import summary
import torch

# 모델 인스턴스 생성
model = VoxelMBConvClassifier(in_ch=63, emb_dim=128, dropout_p=0.3)

# 예시 입력 정의
# voxel feature: [B, 63, 7, 7, 7]
# ref_idx / mut_idx: [B]
batch_size = 4
input_voxel = torch.randn(batch_size, 63, 7, 7, 7)
ref_idx = torch.randint(0, 21, (batch_size,))
mut_idx = torch.randint(0, 21, (batch_size,))

# torchinfo.summary 호출
summary(model, input_data=(input_voxel, ref_idx, mut_idx), 
        col_names=["input_size", "output_size", "num_params"],
        depth=3, 
        device="cpu")

Layer (type:depth-idx)                        Input Shape               Output Shape              Param #
VoxelMBConvClassifier                         [4, 63, 7, 7, 7]          [4, 2]                    --
├─Sequential: 1-1                             [4, 63, 7, 7, 7]          [4, 128, 4, 4, 4]         --
│    └─MBConv3D: 2-1                          [4, 63, 7, 7, 7]          [4, 32, 7, 7, 7]          --
│    │    └─Sequential: 3-1                   [4, 63, 7, 7, 7]          [4, 378, 7, 7, 7]         24,570
│    │    └─Sequential: 3-2                   [4, 378, 7, 7, 7]         [4, 378, 7, 7, 7]         10,962
│    │    └─SqueezeExcitation3D: 3-3          [4, 378, 7, 7, 7]         [4, 378, 7, 7, 7]         11,733
│    │    └─Sequential: 3-4                   [4, 378, 7, 7, 7]         [4, 32, 7, 7, 7]          12,160
│    └─MBConv3D: 2-2                          [4, 32, 7, 7, 7]          [4, 32, 7, 7, 7]          --
│    │    └─Sequential: 3-5                   [4, 32, 7, 7, 7]        

In [ ]:
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VoxelMBConvClassifier(in_ch=63, emb_dim=128, dropout_p=0.3).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

num_epochs = 100
best_pr_auc = 0.0
save_path = "/mnt/e/CAGI_data/best_model_250806_struct.pth"

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for x, ref_idx, mut_idx, y in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        x = x.to(device)               # [B, 63, 7, 7, 7]
        ref_idx = ref_idx.to(device)  # [B]
        mut_idx = mut_idx.to(device)  # [B]
        y = y.to(device)              # [B]

        optimizer.zero_grad()
        logits = model(x, ref_idx, mut_idx)  # [B, 2]
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)

    scheduler.step()
    avg_train_loss = train_loss / len(train_loader.dataset)  

    # --- Validation ---
    model.eval()
    val_loss = 0
    all_preds = []
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for x, ref_idx, mut_idx, y in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            x = x.to(device)
            ref_idx = ref_idx.to(device)
            mut_idx = mut_idx.to(device)
            y = y.to(device)

            logits = model(x, ref_idx, mut_idx)
            loss = criterion(logits, y)

            probs = torch.softmax(logits, dim=1)[:, 1]  # P(class=1)

            val_loss += loss.item() * x.size(0)
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader.dataset)
    pr_auc = average_precision_score(all_labels, all_probs)

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val PR-AUC: {pr_auc:.4f}")

    # --- Save best model ---
    if pr_auc > best_pr_auc:
        best_pr_auc = pr_auc
        torch.save(model.state_dict(), save_path)
        print(f">>> Best model saved! PR-AUC: {pr_auc:.4f}")


Epoch 1 [Val]: 100%|██████████| 241/241 [00:23<00:00, 10.44it/s]



Epoch 1/100
Train Loss: 0.4920 | Val Loss: 0.4706 | Val PR-AUC: 0.7980
>>> Best model saved! PR-AUC: 0.7980


Epoch 2 [Val]: 100%|██████████| 241/241 [00:16<00:00, 14.69it/s]



Epoch 2/100
Train Loss: 0.4728 | Val Loss: 0.4647 | Val PR-AUC: 0.8055
>>> Best model saved! PR-AUC: 0.8055


Epoch 3 [Val]: 100%|██████████| 241/241 [00:18<00:00, 13.27it/s]


...

Epoch 5 [Val]: 100%|██████████| 241/241 [00:18<00:00, 13.35it/s]



Epoch 5/100
Train Loss: 0.4562 | Val Loss: 0.4522 | Val PR-AUC: 0.8262
>>> Best model saved! PR-AUC: 0.8262


Epoch 6 [Val]: 100%|██████████| 241/241 [00:17<00:00, 13.88it/s]



Epoch 6/100
Train Loss: 0.4532 | Val Loss: 0.4490 | Val PR-AUC: 0.8295
>>> Best model saved! PR-AUC: 0.8295


Epoch 7 [Train]:  67%|██████▋   | 1452/2163 [01:47<00:49, 14.50it/s]